In [1]:
pip install huggingface_hub timm torch torchvision

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 13.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import hf_hub_download
import torch
import timm
import torch.nn as nn

c:\Users\Ramanjan Manchikatla\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_path = hf_hub_download(
    repo_id="ramanjanm/efficientnet_mstar",
    filename="efficientnet_mstar.pt"
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\Ramanjan Manchikatla\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ramanjan Manchikatla\.cache\huggingface\hub\models--ramanjanm--efficientnet_mstar. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrat

In [4]:
model = timm.create_model("efficientnet_b0", pretrained=False)

model.classifier = nn.Sequential(
    nn.Linear(1280,256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256,8)
)

In [6]:
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()


EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [7]:
from PIL import Image
from torchvision import transforms

In [8]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [19]:
img = Image.open("HB03333 (another copy).jpg")
img = transform(img).unsqueeze(0)

In [20]:
with torch.no_grad():
    output = model(img)
    _, pred = torch.max(output,1)

print("Predicted class:", pred.item())

Predicted class: 2


In [21]:
classes = [
"2S1",
"BMP2",
"BRDM2",
"BTR60",
"BTR70",
"D7",
"T62",
"ZIL131"
]

In [24]:
print("Predicted vehicle:", classes[pred.item()-1])

Predicted vehicle: BMP2
